# M10: Nerfstudio 3D Reconstruction — Neural Radiance Fields

**Pipeline Position:** Stage 6: Neural Reconstruction — 3D (Nerfstudio; blog uses Omniverse NuRec)  
**Input S3 Path:** `s3://av30lab-shared-data-{account_id}/datasets/nuscenes-mini/` (camera images)  
**Output S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m10/`  
**Instance:** ml.g5.xlarge (1x NVIDIA A10G, 24GB VRAM — ~$1.41/hr)  

> **Note:** This is the open-source alternative to NVIDIA Omniverse NuRec (commercial).  
> Nerfstudio provides equivalent 3D reconstruction capability using open-source tools on standard GPU instances.

> ⚠️ **Known limitation — read before running.** Cells 1–4 (setup, GPU check,
> Nerfstudio install, nuScenes data prep) **run** and demonstrate the reconstruction
> pipeline. The **splatfacto training cell (cell 5) does NOT run** on the current
> SageMaker Distribution GPU image: `gsplat` JIT-compiles its CUDA kernels from
> source and the image lacks a complete CUDA dev toolkit. Real training needs a
> **custom image with the full CUDA toolkit** (or a prebuilt gsplat wheel). Treat
> M10 as an **optional / concept + data-pipeline demo**; the training step is
> deferred. Details + fix options: `docs/TODO_M10_nerfstudio.md`. (There is a fuller
> note right before the training cell.)

## What This Module Does

1. Loads nuScenes multi-camera images from S3 shared data
2. Processes camera poses and intrinsics for 3D reconstruction
3. **Attempts to train** a **splatfacto** model (3D Gaussian Splatting) on a small
   driving scene — ⚠️ **deferred on the current image** (see the note above); the
   runnable part of M10 today is the setup + data-prep pipeline in cells 1–4
4. Renders novel viewpoints from the trained model
5. Displays and exports rendered outputs

## Why 3D Reconstruction for AV?

- **Simulation** — Generate photorealistic 3D environments for testing AV perception
- **Data augmentation** — Render novel viewpoints for training data diversity
- **Scene understanding** — Build 3D maps for planning and prediction validation
- **Digital twins** — Create accurate 3D replicas of real driving environments

In [ ]:
"""Environment Setup"""
import os
import json
import time
import subprocess
import sys
from pathlib import Path
from datetime import datetime, timezone

import boto3
import torch
import numpy as np

# --- S3 Path Configuration ---
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PROFILE = os.environ.get("USER_PROFILE", "default")

USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
INPUT_PREFIX = "datasets/nuscenes-mini/"
OUTPUT_PREFIX = f"users/{PROFILE}/m10/"

# Local paths
LOCAL_DATA_DIR = Path("/tmp/nuscenes-mini")
LOCAL_OUTPUT_DIR = Path("/tmp/m10_output")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Clients ---
s3 = boto3.client("s3")

print(f"Account ID: {ACCOUNT_ID}")
print(f"Profile: {PROFILE}")
print(f"Input: s3://{SHARED_BUCKET}/{INPUT_PREFIX}")
print(f"Output: s3://{USER_BUCKET}/{OUTPUT_PREFIX}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
"""Pre-flight GPU Check — Require 24GB VRAM (g5.xlarge: 1x A10G)"""

REQUIRED_VRAM_GB = 24
RECOMMENDED_INSTANCE = "ml.g5.xlarge"

def check_gpu():
    """Validate GPU availability and VRAM."""
    if not torch.cuda.is_available():
        print("ERROR: No GPU detected!")
        print(f"This notebook requires {REQUIRED_VRAM_GB}GB VRAM.")
        print(f"Recommended instance: {RECOMMENDED_INSTANCE}")
        return False

    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    print("GPU Status:")
    print("=" * 60)

    max_single_vram_mb = 0
    total_vram_mb = 0
    for i, line in enumerate(result.stdout.strip().split("\n")):
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            name, total_mb, free_mb = parts[0], int(parts[1]), int(parts[2])
            total_vram_mb += int(total_mb)
            max_single_vram_mb = max(max_single_vram_mb, int(total_mb))
            print(f"  GPU {i}: {name} | Total: {int(total_mb)/1024:.1f}GB | Free: {int(free_mb)/1024:.1f}GB")

    total_vram_gb = total_vram_mb / 1024
    max_single_vram_gb = max_single_vram_mb / 1024
    # splatfacto trains on ONE GPU — the LARGEST single card must have the
    # VRAM, not the sum across cards (a 4xL4 box has 96GB total but only 24GB
    # per card; a hypothetical 4x8GB box would sum to 32GB yet fit nothing).
    print(f"\nLargest single GPU: {max_single_vram_gb:.1f}GB  (total across GPUs: {total_vram_gb:.1f}GB)")
    print(f"Required (single):  {REQUIRED_VRAM_GB}GB")

    if max_single_vram_gb < REQUIRED_VRAM_GB * 0.9:
        print(f"\nWARNING: No single GPU has {REQUIRED_VRAM_GB}GB "
              f"(largest is {max_single_vram_gb:.1f}GB)")
        print(f"Recommendation: Switch to {RECOMMENDED_INSTANCE}")
        return False

    print("\nGPU check PASSED.")
    return True

gpu_ok = check_gpu()
if not gpu_ok:
    raise RuntimeError(
        f"Insufficient GPU resources. Requires {REQUIRED_VRAM_GB}GB VRAM. "
        f"Use instance: {RECOMMENDED_INSTANCE}"
    )

In [ ]:
"""Install Nerfstudio and dependencies"""

print("Installing Nerfstudio (this may take 3-5 minutes)...")
start_install = time.time()

# Pre-install fpsample from a prebuilt wheel — the latest fpsample (1.0.2)
# fails to compile on this image (pybind11 multiple_interpreters); 0.1.0 ships
# a cp312 manylinux wheel, so --only-binary avoids the source build.
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "fpsample==0.1.0", "--only-binary=:all:", "--quiet"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "nerfstudio",
    "--quiet"
])

# nerfstudio pins gsplat==1.4.0, whose PyPI wheel is pure-Python and JIT-compiles
# its CUDA kernels on first use — which fails on this image because the conda CUDA
# dev packages are incomplete (missing cuda_runtime.h / fatbinary_section.h and a
# matching nvvm layout). setup_gsplat_env.sh installs the missing dev headers,
# symlinks nvvm so nvcc finds cicc, adds the version-matched TF nvcc headers as a
# gap-filler, and source-builds gsplat 1.4.0 (verified on g6/L4 + g5/A10G, CUDA
# 12.9). It is idempotent and EPHEMERAL — re-run after any app restart (the built
# .so lives in site-packages, which resets). ~3-5 min first time, seconds after.
def _find_gsplat_setup():
    for base in [Path.cwd(), Path.cwd().parent, Path.home()]:
        cand = base / "scripts" / "setup_gsplat_env.sh"
        try:
            if cand.exists():
                return str(cand)
        except OSError:
            continue
    return None

_gsplat_setup = _find_gsplat_setup()
if _gsplat_setup is None:
    # Fall back to the copy staged in S3 (notebook-templates ships scripts/ too).
    _gsplat_setup = "/tmp/setup_gsplat_env.sh"
    subprocess.run(
        ["aws", "s3", "cp",
         f"s3://{SHARED_BUCKET}/notebook-templates/scripts/setup_gsplat_env.sh", _gsplat_setup],
        check=True,
    )
print(f"\nBuilding the gsplat CUDA backend via: {_gsplat_setup}")
_gs = subprocess.run(["bash", _gsplat_setup], text=True)
if _gs.returncode != 0:
    raise RuntimeError(
        "gsplat CUDA build failed — the splatfacto training cell cannot run. "
        "Scroll up for the failing step. See docs/TODO_M10_nerfstudio.md for the "
        "fallback (treat M10 as a concept + data-pipeline demo)."
    )

install_time = time.time() - start_install
print(f"Installation complete in {install_time:.1f}s")

# Verify installation
result = subprocess.run(["ns-train", "--help"], capture_output=True, text=True)
if result.returncode == 0:
    print("Nerfstudio CLI verified: ns-train available")
else:
    print(f"WARNING: ns-train not in PATH. Return code: {result.returncode}")

# Confirm nerfstudio is importable and report its version from
# package metadata (the module does not expose __version__).
import importlib.metadata
import nerfstudio  # noqa: F401 — import proves it loads
try:
    _ns_ver = importlib.metadata.version("nerfstudio")
except Exception:
    _ns_ver = "unknown"
print(f"Nerfstudio version: {_ns_ver}")

In [ ]:
"""Load nuScenes camera images and prepare scene data"""

# Download a subset of nuScenes images for 3D reconstruction
print(f"Syncing camera images from S3: s3://{SHARED_BUCKET}/{INPUT_PREFIX}")

# We use CAM_FRONT images for a forward-facing scene reconstruction
cam_prefix = f"{INPUT_PREFIX}samples/CAM_FRONT/"

start_download = time.time()
subprocess.run(
    ["aws", "s3", "sync",
     f"s3://{SHARED_BUCKET}/{cam_prefix}",
     str(LOCAL_DATA_DIR / "images"),
     "--quiet"],
    check=True
)
download_time = time.time() - start_download

# List downloaded images
image_files = sorted((LOCAL_DATA_DIR / "images").glob("*.jpg"))
print(f"Downloaded {len(image_files)} images in {download_time:.1f}s")

# For nerfstudio, we need to create a transforms.json with camera poses
# In production, these come from nuScenes ego_pose + calibration data
# Here we generate approximate poses from sequential frames

def create_transforms_json(image_dir: Path, output_path: Path, num_frames: int = 30):
    """Create a COLMAP-style transforms.json for nerfstudio.

    IMPORTANT: these camera poses are a SYNTHETIC forward-driving trajectory
    (a smooth sin-wave), NOT the real nuScenes calibrated_sensor + ego_pose.
    They let the 3D-training pipeline RUN end-to-end (a smoke test), but the
    reconstruction they produce is NOT geometrically meaningful — a real scene
    needs the actual per-frame extrinsics from nuScenes calibration/LiDAR SLAM.
    """
    images = sorted(image_dir.glob("*.jpg"))[:num_frames]
    
    # nuScenes CAM_FRONT intrinsics (approximate)
    fx, fy = 1266.417, 1266.417  # focal lengths in pixels
    cx, cy = 816.267, 491.507    # principal point
    w, h = 1600, 900             # image dimensions
    
    frames = []
    for i, img_path in enumerate(images):
        # Simulate forward driving: translate along Z axis
        # Small rotations for realism
        t = i / len(images)
        tx = np.sin(t * 0.5) * 0.3  # slight lateral motion
        ty = 0.0
        tz = t * 5.0  # forward motion (5m total)
        
        # Rotation matrix (small yaw variation)
        yaw = np.sin(t * 0.3) * 0.05  # small yaw
        R = np.array([
            [np.cos(yaw), 0, np.sin(yaw), tx],
            [0, 1, 0, ty],
            [-np.sin(yaw), 0, np.cos(yaw), tz],
            [0, 0, 0, 1]
        ])
        
        frames.append({
            "file_path": str(img_path.relative_to(output_path.parent)),
            "transform_matrix": R.tolist()
        })
    
    transforms = {
        "camera_model": "OPENCV",
        "fl_x": fx,
        "fl_y": fy,
        "cx": cx,
        "cy": cy,
        "w": w,
        "h": h,
        "frames": frames
    }
    
    output_path.write_text(json.dumps(transforms, indent=2))
    return transforms

transforms_path = LOCAL_DATA_DIR / "transforms.json"
transforms = create_transforms_json(
    LOCAL_DATA_DIR / "images",
    transforms_path,
    num_frames=min(30, len(image_files))
)

print(f"\nScene setup:")
print(f"  Frames for training: {len(transforms['frames'])}")
print(f"  Image size: {transforms['w']}x{transforms['h']}")
print(f"  Focal length: ({transforms['fl_x']:.1f}, {transforms['fl_y']:.1f})")
print(f"  Transforms file: {transforms_path}")
print("\n  NOTE: poses are SYNTHETIC (sin-wave), not real nuScenes calibration —")
print("  this validates the pipeline runs; the reconstruction is a smoke test,")
print("  not a metrically correct scene. See the markdown note above.")

> ⚙️ **3D training requires a per-session gsplat CUDA build (the install cell handles it).**
>
> The `splatfacto` model uses **gsplat**, which compiles CUDA kernels from source
> on first use (its PyPI wheel is pure-Python). The SageMaker Distribution image
> ships the CUDA runtime + `nvcc` but its conda CUDA dev packages are incomplete
> and split-laid-out, so a naive build fails on missing headers (`cuda_runtime.h`,
> `fatbinary_section.h`) and an `nvvm` mismatch — and `ns-train` re-compiles in a
> fresh subprocess that ignores any per-shell `CPATH`.
>
> **`scripts/setup_gsplat_env.sh` (run by the install cell above) fixes all of it**
> the durable way: it installs the missing dev headers, symlinks `nvvm` so `nvcc`
> finds `cicc`, and **symlinks the real CUDA headers/libs into the standard
> `$CUDA_HOME/include` + `lib64` paths that torch hard-codes** — so the build works
> with zero env vars, whether imported in a terminal or re-JITed inside
> `ns-train`'s subprocess. It also adds the version-matched TensorFlow-bundled
> nvcc headers as a gap-filler.
>
> **Two things to know:**
> - **Ephemeral — this is a per-SESSION bootstrap, not a one-time install.**
>   `/opt/conda` is an image layer, so a JupyterLab app stop/restart wipes the dev
>   headers + symlinks (the gsplat Python package survives, but the CUDA build
>   inputs do not). **Re-run the install cell at the start of every session** before
>   this training cell. It is idempotent (~3–5 min cold, seconds when already built).
> - The camera poses in the previous cell are a **synthetic sin-wave trajectory**,
>   not real nuScenes calibration — so training runs end-to-end (a genuine
>   Gaussian-Splatting pipeline) but the reconstruction is a **smoke test**, not a
>   metrically correct scene. Wiring real `calibrated_sensor` + `ego_pose` into
>   `transforms.json` is the next step for a true reconstruction.
>
> The blog's canonical Stage 6 uses NVIDIA Omniverse **NuRec** (commercial);
> Nerfstudio is the open-source stand-in shown here.

In [ ]:
"""Train splatfacto model (3D Gaussian Splatting)"""

# Training configuration
MODEL_TYPE = "splatfacto"  # 3D Gaussian Splatting — fast training, high quality
MAX_ITERATIONS = 5000     # Reduced for workshop (production: 30000)
OUTPUT_DIR = LOCAL_OUTPUT_DIR / "nerfstudio_output"

print(f"Training {MODEL_TYPE} model:")
print(f"  Max iterations: {MAX_ITERATIONS}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Expected time: 10-15 minutes on g5.xlarge")
print(f"\nStarting training...")

start_train = time.time()

# Launch nerfstudio training via CLI
train_cmd = [
    "ns-train", MODEL_TYPE,
    "--data", str(LOCAL_DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--max-num-iterations", str(MAX_ITERATIONS),
    "--steps-per-eval-image", "500",
    "--steps-per-save", "1000",
    "--pipeline.model.num-downscales", "2",  # Downsample for faster training
    "--viewer.quit-on-train-completion", "True",
    "--vis", "tensorboard",  # Headless: local files only, no viewer/wandb
    "nerfstudio-data",
    "--data", str(LOCAL_DATA_DIR)
]

# Run training process. Disable torch.compile for this subprocess:
# the Triton/inductor backend fails to link libcuda.so on this image,
# and splatfacto runs fine in eager mode (gsplat kernels are prebuilt).
_train_env = {
    **os.environ,
    "TORCH_COMPILE_DISABLE": "1",
    "TORCHDYNAMO_DISABLE": "1",
}
process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=_train_env
)

# Stream output with progress
last_print = time.time()
for line in process.stdout:
    line = line.strip()
    # Print progress updates every 30 seconds
    if "step" in line.lower() or (time.time() - last_print > 30):
        print(f"  {line[:120]}")
        last_print = time.time()

process.wait()
train_time = time.time() - start_train

if process.returncode != 0:
    raise RuntimeError(
        f"ns-train failed (exit {process.returncode}). See streamed "
        f"output above for the cause."
    )

print(f"\nTraining complete!")
print(f"  Duration: {train_time:.1f}s ({train_time/60:.1f} min)")
print(f"  Iterations: {MAX_ITERATIONS}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
"""Render novel views from trained model"""
from PIL import Image
import matplotlib.pyplot as plt

# Find the latest training config
config_files = list(OUTPUT_DIR.rglob("config.yml"))
if not config_files:
    raise FileNotFoundError(f"No config.yml found in {OUTPUT_DIR}")

config_path = sorted(config_files)[-1]  # Most recent
print(f"Using trained model config: {config_path}")

# Render a camera path (orbit around the scene)
RENDER_OUTPUT_DIR = LOCAL_OUTPUT_DIR / "renders"
RENDER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Rendering novel views...")
start_render = time.time()

render_cmd = [
    "ns-render", "interpolate",
    "--load-config", str(config_path),
    "--output-path", str(RENDER_OUTPUT_DIR / "novel_views.mp4"),
    "--pose-source", "train",
    "--interpolation-steps", "10",  # Render 10 interpolated frames
    "--output-format", "images"
]

result = subprocess.run(render_cmd, capture_output=True, text=True)
render_time = time.time() - start_render

if result.returncode == 0:
    print(f"Rendering complete in {render_time:.1f}s")
else:
    print(f"Render completed with warnings (rc={result.returncode})")
    if result.stderr:
        print(f"  stderr: {result.stderr[:200]}")

# Display rendered frames
rendered_images = sorted(RENDER_OUTPUT_DIR.rglob("*.png")) + sorted(RENDER_OUTPUT_DIR.rglob("*.jpg"))
print(f"\nRendered {len(rendered_images)} frames")

if rendered_images:
    # Display a grid of rendered views
    display_count = min(4, len(rendered_images))
    fig, axes = plt.subplots(1, display_count, figsize=(16, 4))
    if display_count == 1:
        axes = [axes]
    
    for i in range(display_count):
        img = Image.open(rendered_images[i * len(rendered_images) // display_count])
        axes[i].imshow(img)
        axes[i].set_title(f"Novel View {i+1}", fontsize=10)
        axes[i].axis("off")
    
    plt.suptitle("Nerfstudio splatfacto — Novel View Synthesis", fontsize=13)
    plt.tight_layout()
    plt.savefig(str(LOCAL_OUTPUT_DIR / "novel_views_grid.png"), dpi=100, bbox_inches="tight")
    plt.show()
else:
    print("No rendered images found — check render output.")
    # Display a training sample instead
    sample_img = next((LOCAL_DATA_DIR / "images").glob("*.jpg"), None)
    if sample_img:
        img = Image.open(sample_img)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.title("Training Input Sample (novel views require full training)")
        plt.axis("off")
        plt.show()

In [ ]:
"""Write rendered outputs and metadata to S3"""

# Upload rendered images
uploaded_files = []

# Upload renders
for render_file in RENDER_OUTPUT_DIR.rglob("*"):
    if render_file.is_file() and render_file.suffix in [".png", ".jpg", ".mp4"]:
        relative = render_file.relative_to(LOCAL_OUTPUT_DIR)
        s3_key = f"{OUTPUT_PREFIX}{relative}"
        s3.upload_file(str(render_file), USER_BUCKET, s3_key)
        uploaded_files.append(s3_key)

# Upload the grid visualization
grid_file = LOCAL_OUTPUT_DIR / "novel_views_grid.png"
if grid_file.exists():
    grid_key = f"{OUTPUT_PREFIX}novel_views_grid.png"
    s3.upload_file(str(grid_file), USER_BUCKET, grid_key)
    uploaded_files.append(grid_key)

# Write metadata
output_metadata = {
    "module": "M10_Nerfstudio_3D_Reconstruction",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model_type": MODEL_TYPE,
    "training": {
        "iterations": MAX_ITERATIONS,
        "training_time_s": round(train_time, 2),
        "num_input_frames": len(transforms["frames"]),
        "image_resolution": f"{transforms['w']}x{transforms['h']}"
    },
    "rendering": {
        "render_time_s": round(render_time, 2),
        "num_rendered_frames": len(rendered_images),
        "output_format": "PNG"
    },
    "uploaded_files": uploaded_files,
    "notes": (
        "Open-source alternative to NVIDIA Omniverse NuRec (commercial). "
        "splatfacto uses 3D Gaussian Splatting for fast training and real-time rendering. "
        "For production AV simulation, use full nuScenes multi-camera setup with "
        "calibrated poses from LiDAR SLAM."
    )
}

metadata_key = f"{OUTPUT_PREFIX}reconstruction_metadata.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=metadata_key,
    Body=json.dumps(output_metadata, indent=2),
    ContentType="application/json"
)
uploaded_files.append(metadata_key)

print(f"Outputs written to S3:")
for f in uploaded_files:
    print(f"  s3://{USER_BUCKET}/{f}")

# Verify
print(f"\nOutput Validation:")
head = s3.head_object(Bucket=USER_BUCKET, Key=metadata_key)
print(f"  OK: {metadata_key} ({head['ContentLength']} bytes)")
print(f"  Total files uploaded: {len(uploaded_files)}")

In [ ]:
"""Cost Analysis — ml.g5.xlarge (~$1.41/hr us-west-2)"""

INSTANCE_TYPE = "ml.g5.xlarge"
INSTANCE_COST_PER_HOUR = 1.41  # USD, ml.g5.xlarge us-west-2 on-demand (~1.408)
KRW_RATE = 1370

# Execution breakdown
setup_min = install_time / 60
data_min = download_time / 60
train_min = train_time / 60
render_min = render_time / 60
total_min = setup_min + data_min + train_min + render_min + 3  # +3 overhead
total_hours = total_min / 60

compute_cost_usd = INSTANCE_COST_PER_HOUR * total_hours
compute_cost_krw = compute_cost_usd * KRW_RATE

print("=" * 60)
print("M10 Nerfstudio 3D Reconstruction — Cost Analysis")
print("=" * 60)
print(f"Instance type:       {INSTANCE_TYPE}")
print(f"Instance cost:       ${INSTANCE_COST_PER_HOUR:.2f}/hr")
print(f"")
print(f"Breakdown:")
print(f"  Nerfstudio install: {setup_min:.1f} min")
print(f"  Data download:      {data_min:.1f} min")
print(f"  Model training:     {train_min:.1f} min ({MAX_ITERATIONS} iterations)")
print(f"  Novel view render:  {render_min:.1f} min")
print(f"  Overhead:           ~3 min")
print(f"  Total time:         {total_min:.1f} min")
print(f"")
print(f"Compute cost:        ${compute_cost_usd:.2f} USD ({compute_cost_krw:.0f} KRW)")
print(f"S3 transfer:         ~$0.01 USD")
print(f"Total:               ${compute_cost_usd + 0.01:.2f} USD ({(compute_cost_usd + 0.01) * KRW_RATE:.0f} KRW)")
print(f"")
print(f"--- vs NVIDIA Omniverse NuRec (Commercial) ---")
print(f"  Omniverse Enterprise: order-of-thousands USD/yr per node (list price varies)")
print(f"  GPU instance (this):  ${INSTANCE_COST_PER_HOUR:.2f}/hr (~${INSTANCE_COST_PER_HOUR * 730:.0f}/mo continuous)")
print(f"  Workshop session:     ${compute_cost_usd:.2f} (one-time)")
print(f"")
print(f"--- Production Scaling (ESTIMATES — not measured) ---")
print(f"  Full scene (30K iter): ~45 min = ${INSTANCE_COST_PER_HOUR * 0.75:.2f} USD")
print(f"  Multi-camera setup:    ~2hr = ${INSTANCE_COST_PER_HOUR * 2:.2f} USD")
print(f"  City-scale (10 scenes): ~${INSTANCE_COST_PER_HOUR * 20:.0f} USD")
print("=" * 60)

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m10-nerfstudio")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")